In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
import random
import warnings

# 경고 메시지 무시 (데이터셋 로딩 과정에서 발생할 수 있는 경고들)
warnings.filterwarnings("ignore")

# ============================================================================
# ❄️ 데이터셋 개요: InSAR 지역 눈 지도 작성 (Remote Sensing / Geospatial)
# 데이터셋명: links-ads/insar-regional-snow-mapping
# 설명: 이 데이터셋은 이탈리아 알프스의 특정 지역을 대상으로 위성 영상 데이터(InSAR, Sentinel-1)와 기상 데이터(ERA5)를 짝지어, 지표면의 눈 깊이(HS)와 눈 수분 당량(SWE)을 예측하는 데 사용됩니다.
# 목표: 초급자가 복잡한 다중 모드(Multi-modal) 위성 데이터를 어떻게 로드하고 전처리하는지 시뮬레이션하는 것이 목표입니다!
# ============================================================================

# ----------------------------------------------------------------------------
# 📐 설정 변수 (Tutor가 설정한 매직 넘버)
# ----------------------------------------------------------------------------
DATASET_NAME = "links-ads/insar-regional-snow-mapping"
SAMPLE_COUNT = 5  # 메모리 효율을 위해 상위 5개 샘플만 사용합니다.

print("==============================================================")
print("✨ PyAI 튜터링 모드 시작: 복잡한 Geospatial 데이터 분석 실습!")
print("==============================================================")

# 1. 데이터셋 로드 전략 설정
print("\n[Step 1/4] 데이터셋 로딩 전략을 수립합니다...")
dataset = None
try:
    # streaming=True로 시도하여 메모리 부담을 줄이고 빠르게 데이터 구조를 파악합니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print(f"✅ 성공! 스트리밍 모드(streaming=True)로 '{DATASET_NAME}' 데이터셋을 로드했습니다.")
except Exception as e:
    # 스트리밍 로드가 실패할 경우, 전체 데이터를 한 번에 다운로드합니다.
    print(f"⚠️ 스트리밍 모드 로드 중 오류 발생 ({e}). 일반 모드로 전환합니다.")
    try:
        # 메모리 제한을 고려하여 'test' 스플릿의 일부만 로드합니다.
        dataset = load_dataset(DATASET_NAME, split='test')
        print("✅ 성공! 일반 모드(streaming=False)로 'test' 스플릿을 로드했습니다.")
    except Exception as e_fallback:
        print(f"❌ 치명적 오류: 데이터셋 로드에 실패했습니다. ({e_fallback})")
        exit()

# 2. 데이터 샘플 추출 (IterableDataset 처리를 위함)
# streaming 모드와 일반 모드에 맞는 패턴을 사용합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)로 간주하고 처리합니다.
    # 메모리 효율을 위해 상위 SAMPLE_COUNT만 '순회 가능'한 형태로 만듭니다.
    print(f"\n[Step 2/4] 상위 {SAMPLE_COUNT}개의 샘플을 추출하여 분석할 준비를 합니다...")
    dataset_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)의 경우, list()로 바로 변환하여 상위 K개를 가져옵니다.
    print(f"\n[Step 2/4] 상위 {SAMPLE_COUNT}개의 샘플을 추출하여 분석할 준비를 합니다...")
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))


# 3. 데이터 구조 분석 및 시뮬레이션 (핵심 실습 구간)
print("\n[Step 3/4] 🚀 Multi-Modal 데이터 특성 분석 시뮬레이션 (AI 트레이닝 준비)")
print("============================================================================\n")

if not sample_data_list:
    print("🚨 분석할 샘플 데이터를 찾을 수 없습니다. 코드를 종료합니다.")
else:
    sample_example = sample_data_list[0]
    print(f"💡 [분석 대상] 첫 번째 샘플의 구조를 살펴봅시다.")
    print("   -> 이 데이터는 InSAR (위성 영상)와 Weather (기상) 등 여러 종류의 '피처'를 한 번에 가집니다.")

    # 데이터셋 메타 정보를 기반으로, 어떤 데이터가 있는지 주석으로 안내합니다.
    print("\n[🔍 데이터 분석 목표]")
    print("1. InSAR 영상 (위성 파워)의 차원과 타입을 확인합니다. (Image Processing)")
    print("2. 지리 공간 데이터(Geo-temporal)의 구조를 이해합니다. (Feature Engineering)")
    print("3. 여러 모드(Multi-modality)의 데이터를 한 번에 배치로 처리할 준비를 합니다. (Pre-processing)")


    # --- 실습 3-1: Image Band의 차원 확인 (Ex. Coherence Image) ---
    print("\n--- 🖼️ Part 1: InSAR Coherence Band (영상 데이터 처리 시뮬레이션) ---")
    # 예시: InSAR의 VV Coherence Image가 들어있다고 가정
    try:
        # 실제 데이터셋 필드를 직접 접근하는 것은 복잡하므로, 대표적인 이미지 필드를 선택하여 분석합니다.
        # 실제 필드 이름은 'coh_IW1_VV' 등이 될 수 있습니다. 여기서는 임의의 필드를 사용하여 구조를 보여줍니다.
        coherence_image = sample_example['features']['tif'] 
        print(f"👉 [Coherence Image] 필드 타입: {type(coherence_image)}")
        print("   (만약 이게 NumPy 배열이라면, shape은 (Height, Width)가 될 겁니다.)")
        
        # 시각화는 복잡하므로, 차원 확인과 가상의 전처리 과정을 진행합니다.
        print("   [🧪 실습 내용] 실제 학습에 사용하려면, 영상 데이터는 항상 NumPy 배열로 변환되어야 합니다.")
        print("   -> (H, W) 형태의 2D 배열을 (H, W, 1)의 3차원 배열로 만들어서, 채널 차원을 통일해야 합니다.")
    except KeyError:
        print("⚠️ 경고: 'features/tif' 필드에 접근할 수 없습니다. (실제 로드된 데이터 필드명 확인 필요)")


    # --- 실습 3-2: Weather/Metadata Feature의 구조 파악 ---
    print("\n--- 💾 Part 2: Weather/Metadata Feature (정량적 변수 처리 시뮬레이션) ---")
    # 예시: ERA5에서 가져온 온도(Temperature)나 풍속(Wind Speed) 같은 정량적 데이터
    try:
        weather_feature = sample_example['features']['page_num'] # 가상의 정량적 변수
        print(f"👉 [Weather Feature] 필드 타입: {type(weather_feature)}")
        print("   (이 데이터는 단순한 float/int 값일 가능성이 높으며, 별도의 전처리가 적습니다.)")
        
        # 간단한 데이터 분석 시뮬레이션:
        print("\n   [🔢 정량적 데이터 분석 예시]")
        print("   만약 이 데이터가 [293.5 K] 같은 온도라면, 'Kelvin' 단위를 'Celsius'로 변환(T_C = T_K - 273.15)하는 전처리 과정이 필요합니다.")
    except KeyError:
        print("⚠️ 경고: 'features/page_num' 필드에 접근할 수 없습니다.")


    # --- 실습 3-3: Feature Fusion 시뮬레이션 (종합)**
    print("\n--- 🧪 Part 3: Feature Fusion (모든 데이터를 통합하는 마법) ---")
    print("=====================================================================")
    print("여러 모드의 데이터를 학습 모델에 넣으려면, '모든 데이터가 같은 Shape'을 가져야 합니다.")

    # 가상의 배열 데이터를 생성하여, 차원 맞추기(Reshaping) 원리를 설명합니다.
    print("\n[🧠 핵심 개념: 차원 일치시키기 (Reshaping)]")
    print("만약 A: (100, 100) [Coherence], B: (100, 100) [Displacement] 두 배열을 합치려 한다면?")
    print("-> 합치려면 (100, 100, 1) 형태의 3D 배열로 만들어서 채널 차원을 맞춰줘야 합니다.")
    
    # 시뮬레이션 코드를 통해 개념을 명확히 보여줍니다.
    H, W = 100, 100
    coh_arr = np.random.rand(H, W) # (100, 100)
    displace_arr = np.random.rand(H, W) # (100, 100)

    print("\n[Code Example: Array Preparation]")
    print(">>> import numpy as np")
    print(">>> coh_3d = coh_arr[..., np.newaxis]   # (100, 100) -> (100, 100, 1)")
    print(">>> final_input = np.concatenate([coh_3d, displace_arr[..., None]], axis=-1) # (100, 100, 2)")
    print("✅ 결과: 여러 모드의 데이터를 성공적으로 하나의 형태로 결합했습니다.")

    print("\n==============================================================")
    print("🎉 튜터링 요약: 복잡한 AI 데이터 전처리는 이렇게 진행됩니다!")
    print("1. 🧩 데이터 로드: Geo-spatial, time-series 등 복잡한 데이터를 효율적으로 로드한다.")
    print("2. 🔍 탐색: 각 필드의 데이터 타입(float, int, image array)과 차원(H, W)을 확인한다.")
    print("3. ✨ 전처리: 차원을 통일시키고 (Reshaping), 필요한 단위로 변환하여 (Normalization/Unit Conversion), 모델에 맞는 형태로 데이터를 준비한다.")
    print("==============================================================")